# 01 — Data Access and Source Verification

**Project:** OULAD — EdTech Learning Analytics  
**Analyst:** Abdeljalil El Khyati  

## Objective
Establish a reproducible path from the official OULAD source to the local environment without keeping raw CSV files in the Git repository.

## Client Perspective
Before analysis, a professional analyst must know:
- Who produced the data.
- What the data represents.
- What tables exist and what one row means in each table (Grain).
- Which fields identify records and how tables relate.
- What limitations are documented.

## Analyst Question
**Question:** Should we first load the entire dataset into Pandas?  
**Answer:** No. The project is SQL-first. Python should control ingestion and inspection, while PostgreSQL stores the persistent working data. We avoid reading millions of rows into memory when a controlled extraction is sufficient.

## 1. Data Provenance

- **Dataset:** Open University Learning Analytics Dataset (OULAD)
- **Primary Publisher:** The Open University
- **Repository:** UCI Machine Learning Repository
- **Dataset ID:** 349
- **DOI:** 10.24432/C5KK69
- **License:** CC BY 4.0

The Open University is the publisher of the OULAD dataset. The UCI Machine Learning Repository provides a stable academic distribution of the dataset. The data contains information about courses, students, and student interactions with the Virtual Learning Environment (VLE), stored in multiple CSV tables connected by identifiers.

### 1.1 Data Contract & Expectations

Before executing any ingestion, we establish a strict **Data Contract** based on the official UCI repository documentation for Dataset 349.

**Expectation:** 
The source archive must contain exactly the following 7 CSV files. Any deviation (missing files, extra files, or renamed files) indicates a corrupted source or a version mismatch, and the pipeline should flag it.

**Expected Files:**
- `courses.csv`
- `assessments.csv`
- `studentInfo.csv`
- `studentRegistration.csv`
- `studentAssessment.csv`
- `vle.csv`
- `studentVle.csv`

**Why this specific code structure?**
1. **Dynamic Path Resolution (`pathlib`):** Ensures the script works reliably regardless of whether Jupyter is launched from the project root or the `notebooks` directory, preventing accidental data scattering.
2. **Idempotency (`if not target_path.exists()`):** The script is designed to be safely re-run. It will skip downloading and extracting files that already exist, saving time, bandwidth, and preventing accidental data corruption or memory overload.

In [1]:
import os
import urllib.request
import zipfile
from pathlib import Path

# --------------------------------------------------
# Configuration & Dynamic Path Resolution (Analyst Decision)
# --------------------------------------------------

current_dir = Path.cwd().resolve()
if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
ZIP_FILE_PATH = RAW_DATA_DIR / "oulad_dataset.zip"

DATA_URL = "https://archive.ics.uci.edu/static/public/349/open+university+learning+analytics+dataset.zip"

EXPECTED_FILES = [
    "courses.csv",
    "assessments.csv",
    "studentInfo.csv",
    "studentRegistration.csv",
    "studentAssessment.csv",
    "vle.csv",
    "studentVle.csv",
]

# --------------------------------------------------
# 1. Prepare raw directory
# --------------------------------------------------
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Starting OULAD source ingestion check...")
print(f"Resolved Project Root: {PROJECT_ROOT}")
print(f"Target Directory: {RAW_DATA_DIR}")

# --------------------------------------------------
# 2. Download archive (Idempotent: Skips if exists)
# --------------------------------------------------
if not ZIP_FILE_PATH.exists():
    print("\nDownloading OULAD dataset from UCI Repository...")
    urllib.request.urlretrieve(DATA_URL, ZIP_FILE_PATH)
    print("Download completed.")
else:
    print("\n[SKIPPED] Archive already exists locally.")

# --------------------------------------------------
# 3. Inspect archive contents
# --------------------------------------------------
print("\n--- Archive Contents ---")
with zipfile.ZipFile(ZIP_FILE_PATH, "r") as zip_ref:
    archive_files = zip_ref.namelist()
    for file_name in archive_files:
        print(f"  - {file_name}")

# --------------------------------------------------
# 4. Extract expected CSV files (Idempotent: Skips extraction if file exists)
# --------------------------------------------------
print("\n--- Extraction ---")
with zipfile.ZipFile(ZIP_FILE_PATH, "r") as zip_ref:
    for file_name in EXPECTED_FILES:
        matching_files = [name for name in archive_files if name.endswith(file_name)]
        
        if matching_files:
            source_file = matching_files[0]
            target_path = RAW_DATA_DIR / file_name
            
            # PROFESSIONAL FIX: Check if file exists before extracting
            if not target_path.exists():
                with zip_ref.open(source_file) as source:
                    with open(target_path, "wb") as target:
                        target.write(source.read())
                print(f"[OK] Extracted: {file_name}")
            else:
                print(f"[SKIPPED] {file_name} already exists.")
        else:
            print(f"[MISSING] {file_name} not found in archive.")

# --------------------------------------------------
# 5. Validate extracted files
# --------------------------------------------------
print("\n--- File Validation ---")
all_present = True

for file_name in EXPECTED_FILES:
    file_path = RAW_DATA_DIR / file_name
    if file_path.exists():
        size_mb = file_path.stat().st_size / (1024 * 1024)
        print(f"[VALID] {file_name:<25} | Size: {size_mb:.2f} MB")
    else:
        print(f"[MISSING] {file_name}")
        all_present = False

# --------------------------------------------------
# 6. Final result
# --------------------------------------------------
if all_present:
    print("\nSUCCESS: All expected OULAD CSV files are available and verified.")
else:
    print("\nERROR: One or more expected files are missing.")

Starting OULAD source ingestion check...
Resolved Project Root: G:\OULAD-EdTech-Learning-Analytics-starter\OULAD-EdTech-Learning-Analytics
Target Directory: G:\OULAD-EdTech-Learning-Analytics-starter\OULAD-EdTech-Learning-Analytics\data\raw

[SKIPPED] Archive already exists locally.

--- Archive Contents ---
  - assessments.csv
  - courses.csv
  - studentAssessment.csv
  - studentInfo.csv
  - studentRegistration.csv
  - studentVle.csv
  - vle.csv
  - OULAD.names

--- Extraction ---
[SKIPPED] courses.csv already exists.
[SKIPPED] assessments.csv already exists.
[SKIPPED] studentInfo.csv already exists.
[SKIPPED] studentRegistration.csv already exists.
[SKIPPED] studentAssessment.csv already exists.
[SKIPPED] vle.csv already exists.
[SKIPPED] studentVle.csv already exists.

--- File Validation ---
[VALID] courses.csv               | Size: 0.00 MB
[VALID] assessments.csv           | Size: 0.01 MB
[VALID] studentInfo.csv           | Size: 3.30 MB
[VALID] studentRegistration.csv   | Size: 1

### 1.2 Contract Validation & Execution Result

**Validation:** 
The execution output confirms that the **Data Contract was fully satisfied**. 
- The archive was successfully located (or downloaded).
- All 7 expected files were found within the archive.
- The idempotency check successfully skipped re-extraction of existing files (if applicable).
- File sizes match the expected magnitude (e.g., `studentVle.csv` is ~454 MB, confirming it is the large interaction table).

**Decision:** 
The source data is verified and structurally sound. We can safely proceed to the Profiling stage.

## 2. Output Interpretation

The ingestion step is considered successful when:
1. The UCI archive is downloaded successfully.
2. The archive contents can be inspected.
3. The expected OULAD CSV files are extracted into `data/raw/`.
4. All expected files are physically present.
5. File sizes are recorded for traceability.

No CSV file is loaded into Pandas at this stage. The purpose of this step is source verification and controlled ingestion. Data profiling will be performed in the next stage.

## 3. Source Metadata

The archive contains the seven expected CSV data tables and an additional `OULAD.names` metadata file.

The metadata file is not part of the analytical tables, but it may contain useful source definitions and will be retained as a reference for data interpretation.

In [2]:
from pathlib import Path

print("\n--- RAW DATA DIRECTORY CONTENTS (Final State) ---")

for path in sorted(RAW_DATA_DIR.iterdir()):
    if path.is_file():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"{path.name:<30} {size_mb:>10.2f} MB")


--- RAW DATA DIRECTORY CONTENTS (Final State) ---
assessments.csv                      0.01 MB
courses.csv                          0.00 MB
oulad_dataset.zip                   44.58 MB
studentAssessment.csv                5.43 MB
studentInfo.csv                      3.30 MB
studentRegistration.csv              1.08 MB
studentVle.csv                     432.81 MB
vle.csv                              0.26 MB


## 4. Ingestion Verification

The extracted files are now present in the project-level `data/raw/` directory.

The archive also contains `OULAD.names`, which is treated as source metadata rather than an analytical table.

The raw CSV files remain outside Git because the project `.gitignore` excludes the `data/` directory.